In [1]:
%pip install -q praw requests beautifulsoup4

Note: you may need to restart the kernel to use updated packages.


## Configuration

In [2]:
import re
import json
import time
import html
from pathlib import Path
from bs4 import BeautifulSoup

CHECKPOINT_PATH  = Path("vi_qa_crawled.jsonl")
MIN_SCORE        = 20
MAX_AGE_YEARS    = 10
TARGET           = 1000
MAX_COMMENTS     = 100
MIN_ANSWER_WORDS = 30   

SUBREDDITS = ["vozforums", "TroChuyenLinhTinh", "VietNamNation", "VietTalk"]

QUESTION_PATTERNS = [
    r"\?",
    r"^không",
    r"^nhỉ",
    r"^ai ",
    r"^có ai ",
    r"^cho mình hỏi",
    r"^làm sao",
    r"^tại sao",
    r"^à",
    r"^sao",
    r"^là gì",
    r"^vậy",
]

def is_question(text: str) -> bool:
    if not text or not isinstance(text, str):
        return False
    t = text.strip().lower()
    return any(re.search(p, t, re.IGNORECASE) for p in QUESTION_PATTERNS)

def clean_text(text: str) -> str:
    """
    Remove HTML tags, unescape entities, and normalize whitespace.
    """
    if not text or not isinstance(text, str):
        return ""
    # Unescape HTML entities (&amp; &lt; &gt; &quot; etc.)
    text = html.unescape(text)
    # Strip HTML tags
    soup = BeautifulSoup(text, "html.parser")
    text = soup.get_text(separator=" ", strip=True)
    # Collapse multiple whitespace (spaces, newlines, tabs)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

Checkpoint helpers


In [3]:
def load_checkpoint():
    """Return (records, seen_post_ids, seen_questions) from checkpoint file."""
    if not CHECKPOINT_PATH.exists():
        print("No checkpoint found. Starting fresh.")
        return [], set(), set()
    records, seen_ids, seen_questions = [], set(), set()
    with open(CHECKPOINT_PATH, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rec = json.loads(line)
                records.append(rec)
                seen_ids.add(rec.get("post_id", ""))
                seen_questions.add(rec.get("question", "").strip().lower())
    print(f"Loaded {len(records)} existing records from checkpoint.")
    return records, seen_ids, seen_questions

def save_record(record: dict, records: list, seen_ids: set, seen_questions: set):
    """Append one record to the JSONL file and update all in-memory sets."""
    with open(CHECKPOINT_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
    records.append(record)
    seen_ids.add(record["post_id"])
    seen_questions.add(record["question"].strip().lower())

Reddit JSON helpers

In [4]:
import requests

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; rv:109.0) Gecko/20100101 Firefox/109.0"
}

def get_top_posts(subreddit: str, after: str = None):
    """
    Fetch one page (up to 100) of top posts from a subreddit.
    Returns (posts_list, next_after).
    """
    params = {"sort": "top", "t": "all", "limit": 100}
    if after:
        params["after"] = after
    url = f"https://old.reddit.com/r/{subreddit}/top.json"
    try:
        resp = requests.get(url, headers=HEADERS, params=params, timeout=15)
        if resp.status_code != 200:
            print(f"  HTTP {resp.status_code} for r/{subreddit}")
            return [], None
        data = resp.json()["data"]
        return data["children"], data.get("after")
    except Exception as e:
        print(f"  Error fetching posts: {e}")
        return [], None

def get_best_comment(subreddit: str, post_id: str):
    """
    Fetch top-level comments for a post and return the highest-scored one.
    Returns comment body string or None.
    """
    url = f"https://old.reddit.com/r/{subreddit}/comments/{post_id}.json"
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        if resp.status_code != 200:
            return None
        comments_root = resp.json()[1]["data"]["children"]
        candidates = []
        for item in comments_root[:MAX_COMMENTS]:
            if item["kind"] != "t1":
                continue
            body = clean_text(item["data"].get("body", ""))
            score = item["data"].get("score", 0)
            if not body or body in ("[deleted]", "[removed]") or len(body.split()) < MIN_ANSWER_WORDS:
                continue
            candidates.append((score, body))
        if not candidates:
            return None
        best_score, best_body = max(candidates, key=lambda x: x[0])
        if best_score < 1:
            return None
        return best_body
    except Exception as e:
        print(f"  Error fetching comments for {post_id}: {e}")
        return None

Crawler loop

In [5]:
from datetime import datetime, timezone

# Load existing progress
records, seen_ids, seen_questions = load_checkpoint()
five_years_ago = datetime.now(timezone.utc).timestamp() - (MAX_AGE_YEARS * 365.25 * 24 * 3600)

# Progress counters
posts_examined = 0
posts_saved = len(records)  # already have this many from checkpoint

for subreddit_name in SUBREDDITS:
    if len(records) >= TARGET:
        break

    print(f"\n── r/{subreddit_name} ──")
    after = None
    sub_posts = 0
    sub_saved = 0

    while len(records) < TARGET:
        posts, after = get_top_posts(subreddit_name, after=after)
        if not posts:
            print(f"  No more posts from r/{subreddit_name}.")
            break

        for post in posts:
            if len(records) >= TARGET:
                break

            p = post["data"]
            post_id    = p.get("id", "")
            title      = p.get("title", "")
            selftext = (p.get("selftext") or "").strip()
            question = f"{title}\n{selftext}".strip() if selftext else title
            question = clean_text(question)

            score      = p.get("score", 0)
            created    = p.get("created_utc", 0)
            num_comments = p.get("num_comments", 0)

            posts_examined += 1
            sub_posts += 1

            # Filters
            if not question:
                continue
            if not question:
                continue
            if post_id in seen_ids:
                continue
            if score < MIN_SCORE:
                continue
            if created < five_years_ago:
                continue
            if num_comments == 0:
                continue
            if not is_question(title):
                continue
            if title.strip().lower() in seen_questions:
                continue

            best_body = get_best_comment(subreddit_name, post_id)
            if not best_body:
                continue

            record = {
                "question":      question,
                "human_answers": [best_body],
                "source":        f"reddit_{subreddit_name.lower()}",
                "post_id":       post_id,
            }
            save_record(record, records, seen_ids, seen_questions)
            sub_saved += 1

            # Progress: every 10 new records or every 50 posts examined
            if sub_saved % 10 == 1 or posts_examined % 50 == 0:
                pct = 100 * len(records) / TARGET
                print(f"  [{len(records)}/{TARGET}] ({pct:.1f}%) | posts examined: {posts_examined} | r/{subreddit_name}: {sub_saved} saved")

            time.sleep(2.5)  # Reddit rate limit: ~30 req/min for unauthenticated

        if after is None:
            print(f"  r/{subreddit_name} done. Examined {sub_posts} posts, saved {sub_saved}.")
            break

print(f"\nDone. Total: {len(records)} QA pairs from {posts_examined} posts examined.")

Loaded 292 existing records from checkpoint.

── r/vozforums ──


/tmp/ipykernel_2923/1643136440.py:47: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  soup = BeautifulSoup(text, "html.parser")


  r/vozforums done. Examined 499 posts, saved 0.

── r/TroChuyenLinhTinh ──
  r/TroChuyenLinhTinh done. Examined 498 posts, saved 0.

── r/VietNamNation ──
  r/VietNamNation done. Examined 499 posts, saved 0.

── r/VietTalk ──
  [293/1000] (29.3%) | posts examined: 1768 | r/VietTalk: 1 saved


/tmp/ipykernel_2923/1643136440.py:47: MarkupResemblesLocatorWarning: The input looks more like a URL than markup. You may want to use an HTTP client like requests to get the document behind the URL, and feed that document to Beautiful Soup.
  soup = BeautifulSoup(text, "html.parser")


  r/VietTalk done. Examined 429 posts, saved 9.

Done. Total: 301 QA pairs from 1925 posts examined.


Load and display the final HC3-style dataset

In [6]:
from datasets import Dataset
from pprint import pprint

def load_vi_qa_dataset(path=CHECKPOINT_PATH):
    """Read JSONL checkpoint and return an HC3-style HuggingFace Dataset."""
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rec = json.loads(line)
                rows.append({
                    "question":      rec["question"],
                    "human_answers": rec["human_answers"],
                    "source":        rec["source"],
                })
    return Dataset.from_list(rows)

hc3_vi_raw = load_vi_qa_dataset()
print(hc3_vi_raw)
print("\n--- Example VI sample ---")
pprint(hc3_vi_raw[0], width=100)

Dataset({
    features: ['question', 'human_answers', 'source'],
    num_rows: 301
})

--- Example VI sample ---
{'human_answers': ['Con gái bác xinh đẹp mà thức thời nữa. Đừng để thằng Bắc Kỳ hay ôn con gốc '
                   'Thanh Nghệ Tĩnh nào để ý nhé. Chúng nó sẽ kéo con gái bác xuống vũng bùn bằng '
                   'độ ngu của bò đỏ, thói gia trưởng và victim blaming đó.'],
 'question': 'CON GÁI T 13 TUỔI NGHĨ NTN VỀ VN? NHÂN TRONG NƯỚC ĐANG RỘ TRÀO LƯU KIỂU BẦY ĐÀN “TỰ '
             'HÀO LÀ NGƯỜI VN” T NHẮN TIN HỎI CON GÁI XEM NÓ NGHĨ NTN…. VÀ VỚI CHÚT KIẾN THỨC NON '
             'NỚT CỦA ĐỨA 13T, NÓ ĐÃ KHÔNG KHIẾN T PHẢI THẤT VỌNG…',
 'source': 'reddit_vietnamnation'}
